In [1]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import os
import numpy as np
from tensorflow.keras import layers
from IPython.display import Audio
import wave
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

2024-10-27 19:10:16.753856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-27 19:10:16.764786: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-27 19:10:16.768301: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-27 19:10:16.777576: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-27 19:10:17.366549: W tensorflow/compiler/tf2

In [2]:
df = pd.read_csv("../data_processing/sep28k-mfcc.csv")

In [3]:
df = df[df['NaturalPause'] == 0]
df = df[df['Interjection'] == 0]
df = df[df['Prolongation'] == 0]
df = df[df['WordRep'] == 0]
df = df[df['SoundRep'] == 0]
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
2,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
4,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
6,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
9,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
11,HeStutters,0,11,42928800,42976800,0,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [4]:
df = df.reset_index()
df.head()

,index,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,...,29,30,31,32,33,34,35,36,37,38
0,2,HeStutters,0,2,34809760,34857760,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,4,HeStutters,0,4,35721920,35769920,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,6,HeStutters,0,6,37251200,37299200,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,9,HeStutters,0,9,41417440,41465440,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,11,HeStutters,0,11,42928800,42976800,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [5]:
df = df.drop(columns=['index'])
df.head()

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,...,29,30,31,32,33,34,35,36,37,38
0,HeStutters,0,2,34809760,34857760,0,0,0,0,0,...,-3.249218,-3.218731,0.360520,-0.115838,1.723229,-1.460772,-1.430723,-1.429669,-0.178196,-3.046811
1,HeStutters,0,4,35721920,35769920,0,0,0,0,0,...,-2.057481,-3.225782,-1.561315,-3.723942,-2.249673,-3.513439,-2.097951,-1.940040,0.020210,-0.556461
2,HeStutters,0,6,37251200,37299200,0,0,0,0,0,...,-4.985494,-6.181957,-2.396787,-6.544606,-2.319257,-3.688410,-0.221778,-1.690031,0.664374,-0.303436
3,HeStutters,0,9,41417440,41465440,0,0,0,0,0,...,-4.798577,-6.274436,-4.889234,-5.795599,-4.400376,-4.176473,-0.957432,-3.210292,0.577023,1.042720
4,HeStutters,0,11,42928800,42976800,0,0,0,0,0,...,-3.798326,-6.705790,-1.544637,-6.356187,-0.987506,-3.107216,-1.422836,-3.615824,-1.171095,-1.937332


In [6]:
df.to_csv("block.csv",index=False)

In [7]:
import os

def list_files(directory):
    return [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

# Replace 'directory_path' with the path to your directory
directory_path = '/home/alien/Git/DATA/mel_spects_block'
# names_list = list_files(directory_path)
names_list = pd.read_csv("block.csv")['Name'].values.tolist()
# full_names_list = ["/home/alien/Git/DATA/mfcc_images/" + img + ".jpg" for img in names_list]
full_names_list = []
for img in names_list:
    corresponding_sound = df.loc[df['Name'] == img, 'Block'].values[0]
    if corresponding_sound == 0:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_block/" + img + "_fluent.jpg")
    if corresponding_sound >= 1:
        full_names_list.append("/home/alien/Git/DATA/mel_spects_block/" + img + "_stutter.jpg")

print(full_names_list[-2])

/home/alien/Git/DATA/mel_spects_block/WomenWhoStutter_109_33_fluent.jpg


In [8]:
def load_all(imagefile_list):
    data = []
    labels = []

    for imagefile in imagefile_list:
        print(imagefile)
        image = cv2.imread(imagefile)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        data.append(image)

        if "fluent" in imagefile:
            labels.append(0)
        else:
            labels.append(1)

    labels = np.array(labels)
    data = np.array(data)

    return data, labels

In [9]:
X, y = load_all(full_names_list)

/home/alien/Git/DATA/mel_spects_block/HeStutters_0_2_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_4_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_6_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_9_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_11_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_12_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_14_stutter.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_15_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_16_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_20_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_22_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_24_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_26_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_29_fluent.jpg
/home/alien/Git/DATA/mel_spects_block/HeStutters_0_32_fluent.jpg
/home/alien/Git/DATA/mel_spe

In [10]:
print(X[0])
print(y[0])

[[[ 68  33  60]
  [ 96  28  68]
  [ 89  27  67]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[114  57  88]
  [166  57 117]
  [157  55 112]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 [[140  60  94]
  [190  55 118]
  [198  65 126]
  ...
  [  0   0   4]
  [  0   0   4]
  [  0   0   2]]

 ...

 [[126  65 103]
  [194  56 123]
  [189  69 128]
  ...
  [ 29  13  72]
  [ 18   5  59]
  [ 21   6  57]]

 [[ 90  48 102]
  [145  31 116]
  [141  39 122]
  ...
  [ 26  18  68]
  [ 24  17  63]
  [ 27  19  61]]

 [[ 86  33  88]
  [122  36 120]
  [134  36 131]
  ...
  [ 23  14  63]
  [ 27  18  70]
  [ 26  14  66]]]
0


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.3, random_state=9)

In [12]:
print("x_train shape:", X_train.shape)
print("x_test shape:", X_test.shape)
print('y_train shape:', y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (1829, 224, 224, 3)
x_test shape: (784, 224, 224, 3)
y_train shape: (1829,)
y_test shape: (784,)


In [13]:
print(len(X_train), len(X_test), len(y_train), len(y_test))

1829 784 1829 784


In [14]:
from collections import Counter
print(Counter(y_train))
print(Counter(y_test))

Counter({0: 1223, 1: 606})
Counter({0: 475, 1: 309})


In [15]:
print(X_train)

[[[[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  [[  0   0   2]
   [  0   0   2]
   [  0   0   2]
   ...
   [  0   0   4]
   [  0   0   4]
   [  0   0   2]]

  ...

  [[ 11   7  33]
   [ 12   3  51]
   [ 33  16  81]
   ...
   [ 36  15  90]
   [ 36  16  87]
   [ 35  14  83]]

  [[ 23  14  52]
   [ 37  23  85]
   [ 44  22  99]
   ...
   [ 38  19  92]
   [ 38  20  91]
   [ 39  18  84]]

  [[ 30  18  59]
   [ 40  18  88]
   [ 47  17 104]
   ...
   [ 38  17  89]
   [ 41  17  91]
   [ 40  15  88]]]


 [[[ 29  12  42]
   [ 35  11  78]
   [ 41  13  69]
   ...
   [  7   0  15]
   [ 30   6  65]
   [ 27   9  61]]

  [[ 49  20  65]
   [ 65  21 112]
   [ 68  17 101]
   ...
   [  6   0  21]
   [ 52  21  92]
   [ 50  17  90]]

  [[ 59  19  81]
   [ 79  19 129]
   [101  33 134]
   ...
   [ 18   1  43]
   [ 56  22 100]
   [ 56  13

In [16]:
from PIL import Image
IMAGE_DIR = "/home/alien/Git/DATA/mel_spects_block/"

def get_image_dimensions(image_path):
    with Image.open(image_path) as img:
        width, height = img.size
    return width, height

image_path = '/home/alien/Git/DATA/mel_spects_block/HeStutters_0_0_fluent.jpg'  # Change this to the path of your image file
width, height = get_image_dimensions(image_path)
print("Image width:", width)
print("Image height:", height)

Image width: 610
Image height: 450


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, MultiHeadAttention
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dropout, Flatten, Dense, Input, AveragePooling2D, Attention, Reshape, TimeDistributed, Bidirectional, LSTM, GRU
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input, decode_predictions
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.constraints import ClipValue
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Layer, MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization

In [18]:
# def build_model(input_shape=(224, 224, 3)):
#     base_model = VGG19(weights='imagenet', include_top=True, input_tensor=Input(shape=input_shape))
#     # Get the output of the 'fc2' layer in VGG16
#     a = base_model.get_layer('fc2').output

#     # Add a Dense layer with 13 neurons
#     dense_layer = Dense(13, activation='relu')(a)

#     # Add a final output layer with sigmoid activation
#     output_layer = Dense(1, activation='sigmoid')(dense_layer)

#     # Define the model with VGG16 base and the added layers
#     model = Model(inputs=base_model.input, outputs=output_layer)

#     # Freeze the weights of the VGG16 layers
#     for layer in base_model.layers:
#         layer.trainable = False

#     return model
def build_model(input_shape=(224, 224, 3), use_float16=False):
    inputs = Input(shape=input_shape)

    # Convert to lower precision if specified
    if use_float16:
        x = tf.keras.layers.Lambda(lambda t: tf.cast(t, tf.float16))(inputs)
    else:
        x = inputs

    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)
    
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = TimeDistributed(Flatten())(x)  # Flatten along the time dimension
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Attention()([x, x])  # Self-attention mechanism

    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)  # Adding dropout with a rate of 0.5
    
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    return model

In [19]:
  # base_model = VGG19(weights='imagenet', include_top=True,
  #                   input_tensor=Input(shape=(224, 224, 3)))
  # base_model.summary()

In [20]:
vgg_model = build_model()
vgg_model.summary()

I0000 00:00:1730070621.761099   15656 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730070621.780043   15656 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730070621.782016   15656 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730070621.785065   15656 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 112, 112,  │        256 │ max_pooling2d[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,856 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ max_pooling2d_1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    295,168 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │      1,024 │ max_pooling2d_2[… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 28, 7168)  │          0 │ batch_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 28, 256)   │  7,472,128 │ time_distributed… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 28, 256)   │          0 │ bidirectional[0]… │
│ (Attention)         │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 7168)      │          0 │ attention[0][0] 

 Total params: 12,355,649 (47.13 MB)

 Trainable params: 12,354,753 (47.13 MB)

 Non-trainable params: 896 (3.50 KB)

In [21]:
vgg_model.compile(
  optimizer=Adam(0.01),
  loss='binary_crossentropy',
  metrics=['accuracy']
)

In [22]:
batch_size = 40
history = vgg_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=60,
    callbacks=[
        ReduceLROnPlateau(
            monitor = 'accuracy',
            factor = 0.2,
            patience = 5,
            verbose = 1,
            min_lr = 0.0001
        ),
        # early_stop
    ]
)

Epoch 1/60


2024-10-27 19:10:25.760760: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907
W0000 00:00:1730070625.834510   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.855388   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.856950   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.858505   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.863489   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.872006   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.876094   15796 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070625.878243   15796 gpu_t

45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.5980 - loss: 3.7865

W0000 00:00:1730070638.129097   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.130321   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.131489   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.132696   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.134296   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.136608   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.138913   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.140493   15797 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070638.142386   15797 gp

46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 218ms/step - accuracy: 0.5988 - loss: 3.7106 - learning_rate: 0.0100
Epoch 2/60


W0000 00:00:1730070641.855609   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.858356   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.860849   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.864219   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.867247   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.871270   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.876415   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.880245   15790 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1730070641.883795   15790 gp

46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 135ms/step - accuracy: 0.6684 - loss: 0.6828 - learning_rate: 0.0100
Epoch 3/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 135ms/step - accuracy: 0.6630 - loss: 0.6686 - learning_rate: 0.0100
Epoch 4/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6759 - loss: 0.6411 - learning_rate: 0.0100
Epoch 5/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6668 - loss: 0.6323 - learning_rate: 0.0100
Epoch 6/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6544 - loss: 0.6478 - learning_rate: 0.0100
Epoch 7/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6573 - loss: 0.6396 - learning_rate: 0.0100
Epoch 8/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6757 - loss: 0.6432 - learning_rate: 0.0100
Epoch 9/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.6914 - loss: 0.6204 - learning_rate: 0.0100
Epoch 10/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 135ms/step - accuracy: 0.6570 - loss: 0.6441 - learning_rate: 0.0100
Epoch 11/60
46/46 ━━

In [25]:
# predictions
vgg_pred = vgg_model.predict(X_test, batch_size=1)

vgg_pred = np.round(vgg_pred)

  # model evaluation
confusion = confusion_matrix(y_test, vgg_pred)
print(classification_report(y_test, vgg_pred))
print(confusion)

784/784 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
              precision    recall  f1-score   support

           0       0.61      1.00      0.75       475
           1       0.00      0.00      0.00       309

    accuracy                           0.61       784
   macro avg       0.30      0.50      0.38       784
weighted avg       0.37      0.61      0.46       784

[[475   0]
 [309   0]]


/home/alien/Programming/env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/alien/Programming/env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/alien/Programming/env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

In [27]:
vgg_model.save('./model_block.keras', overwrite=True)

: 